# Welsh ASR — XLS-R fine-tuning on Colab

Runs either in browser Colab or through the official Google Colab extension for
VS Code / Cursor. Both execute on the same free T4; the extension just keeps the
notebook in your repo instead of the browser.

Either way the runtime is a separate machine that cannot see your local files,
which is why cell 3 clones the repo.

**Secrets** are read from Colab Secrets when present, otherwise prompted for.
Nothing is written into the notebook. You need `GH_TOKEN` (repo scope),
`HF_TOKEN` (write) and `WANDB_API_KEY`.

**GPU:** browser -> Runtime -> Change runtime type -> T4.
Extension -> kernel picker (top right) -> Colab -> T4.

Target to beat: zero-shot Whisper-small, **WER 0.5987** on FLEURS Welsh test.

In [1]:
print("alive")

alive


## 1. Confirm a GPU was actually allocated

In [1]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
# Free tier sometimes hands you a CPU-only runtime. If this errors, stop and
# retry later rather than training on CPU.

name, memory.total [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 0 MiB


## 2. Install dependencies

In [2]:
!pip install -q "transformers>=5.0" "datasets>=5.0" jiwer accelerate soundfile librosa wandb
# Colab ships transformers 4.x. This upgrade matters: 5.x renamed the flags
# train.py uses. If imports misbehave afterwards, Runtime -> Restart session.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 122.7 MB/s eta 0:00:00


## 3. Get the code

The repo is private, so cloning needs a GitHub token with `repo` scope stored in
Colab Secrets as `GH_TOKEN`. If you have made the repo public, the fallback
clone works without one.

In [3]:
import os, getpass

def secret(name, prompt=None):
    """Fetch a secret from Colab Secrets, the environment, or an interactive prompt.

    google.colab.userdata only exists in the browser UI; under the VS Code /
    Cursor extension it is absent, so fall back to getpass, which also keeps the
    value out of the saved notebook.
    """
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return getpass.getpass(prompt or f"{name}: ")


import subprocess

REPO = "paarthN/welsh-asr"
DIR = "/content/welsh-asr"
tok = secret("GH_TOKEN", "GitHub token (repo scope): ")
url = f"https://{tok}@github.com/{REPO}.git"

# Clone on a fresh runtime, otherwise pull. Skipping outright when the
# directory exists would silently keep running whatever code was fetched
# during the previous session.
if os.path.exists(DIR):
    subprocess.run(["git", "-C", DIR, "remote", "set-url", "origin", url], check=True)
    subprocess.run(["git", "-C", DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", url, DIR], check=True)

%cd /content/welsh-asr
!git log --oneline -1

/content/welsh-asr
8682608 (HEAD -> main, origin/main, origin/HEAD) Fail loudly when a checkpoint does not reach disk


## 4. Secrets and W&B

In [4]:
os.environ["HF_TOKEN"] = secret("HF_TOKEN", "Hugging Face token (write): ")
os.environ["WANDB_API_KEY"] = secret("WANDB_API_KEY", "W&B API key: ")
os.environ["WANDB_PROJECT"] = "welsh-asr"

import wandb; wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: paarth-nawani (paarth-nawani-ucsd) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 5. Mount Drive for checkpoints

This is what makes a disconnect survivable. Each checkpoint is ~3.6GB (model
plus optimizer state) and `save_total_limit=2` keeps two, so budget ~7GB of the
free 15GB.

In [5]:
# Drive gives crash-resilient checkpoints. It is available in browser Colab;
# in the VS Code extension it may not be, so fall back to the runtime's local
# disk. Local disk is wiped when the session ends, so if Drive is unavailable
# treat the run as needing to finish in one session.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_DIR = "/content/drive/MyDrive/welsh-asr-xlsr"
except Exception as e:
    print(f"Drive unavailable ({type(e).__name__}); using local disk.")
    print("WARNING: checkpoints will NOT survive the session ending.")
    CKPT_DIR = "/content/welsh-asr-xlsr"

os.makedirs(CKPT_DIR, exist_ok=True)
print("checkpoints ->", CKPT_DIR)

Mounted at /content/drive
checkpoints -> /content/drive/MyDrive/welsh-asr-xlsr


## 6. Build the vocabulary and processor (CPU, seconds)

In [6]:
!cd src && python prepare_data.py

README.md: 100% 386k/386k [00:00<00:00, 214MB/s]

parquet-data/cy_gb/train-00000-of-00001.(…): downloading bytes:   1% 37.9M/2.74G [00:02<01:26, 31.4MB/s, 2.00MB/s  ]
parquet-data/cy_gb/train-00000-of-00001.(…): downloading bytes:   2% 51.0M/2.74G [00:03<01:00, 44.5MB/s, 3.50MB/s  ]
parquet-data/cy_gb/train-00000-of-00001.(…): reconstructing file:   0% 10.2M/2.74G [00:03<12:07, 3.75MB/s,  538kB/s  ]
parquet-data/cy_gb/train-00000-of-00001.(…): reconstructing file:   1% 17.4M/2.74G [00:03<06:10, 7.35MB/s, 1.13MB/s  ]
parquet-data/cy_gb/train-00000-of-00001.(…): reconstructing file:   1% 22.9M/2.74G [00:03<04:09, 10.9MB/s, 1.59MB/s  ]
parquet-data/cy_gb/train-00000-of-00001.(…): reconstructing file:   1% 27.6M/2.74G [00:03<03:27, 13.1MB/s, 2.16MB/s  ]
parquet-data/cy_gb/train-00000-of-00001.(…): reconstructing file:   1% 32.0M/2.74G [00:03<02:43, 16.6MB/s, 2.47MB/s  ]
parquet-data/cy_gb/train-00000-of-00001.(…): downloading bytes:   3% 69.0M/2.74G [00:04<01:53, 23.5MB/s, 5.59MB/s  ] ]
pa

In [8]:
!cd src && python -u train.py \
    --output-dir "{CKPT_DIR}" \
    --max-steps 2100 --batch-size 2 --grad-accum 8 \
    --save-only-model --save-total-limit 1 \
    --hub-model-id pnawani/welsh-asr-xlsr-300m \
    --push-to-hub --resume

checkpoint-1200: INCOMPLETE {'optimizer.pt'}
checkpoint-400: OK
checkpoint-800: INCOMPLETE {'optimizer.pt'}


In [9]:
import shutil, os
t, u, f = shutil.disk_usage("/content/drive/MyDrive")
print(f"Drive free: {f/1e9:.1f} GB of {t/1e9:.1f} GB")
p = os.path.join(CKPT_DIR, "broken-checkpoint-800")
if os.path.exists(p):
    shutil.rmtree(p); print("removed broken-checkpoint-800")
for c in sorted(os.listdir(CKPT_DIR)):
    d = os.path.join(CKPT_DIR, c)
    if os.path.isdir(d):
        sz = sum(os.path.getsize(os.path.join(d,x)) for x in os.listdir(d))
        print(f"  {c}: {sz/1e9:.2f} GB")

Drive free: 1.0 GB of 16.1 GB
removed broken-checkpoint-800
  checkpoint-1200: 0.00 GB
  checkpoint-400: 3.75 GB
  checkpoint-800: 0.00 GB


## 7. Train

**Start at batch size 2.** Measured locally, batch 4 with 30s clips allocated
27GB in fp32; halved for fp16 that is still ~13.5GB of activations plus ~5GB of
model and optimizer state, which does not fit a 16GB T4.

Step DOWN a row only if CUDA reports OOM. Effective batch stays 16 throughout,
so the learning dynamics do not shift between rows.

| Try | `--batch-size` | `--grad-accum` | `--gradient-checkpointing` |
|---|---|---|---|
| 1 | 2 | 8 | off |
| 2 | 2 | 8 | on |
| 3 | 2 | 8 | on, plus `--max-duration 20` |
| 4 | 1 | 16 | on, plus `--max-duration 20` |

If try 1 leaves plenty of headroom in `nvidia-smi`, `--batch-size 4
--grad-accum 4 --gradient-checkpointing` will run faster.

`max_steps=2100` is about 12 epochs over the 2784 usable clips.

**Watch for:** loss falling from ~19 and continuing down; eval WER starting at
1.0 and dropping below 0.5 within a few evals. If eval WER is still 1.0 after
~1000 steps, stop — that is a vocab problem, not impatience.

In [ ]:
!cd src && python -u train.py \
    --output-dir "{CKPT_DIR}" \
    --max-steps 2100 \
    --batch-size 2 \
    --grad-accum 8 \
    --hub-model-id pnawani/welsh-asr-xlsr-300m \
    --push-to-hub

Filter: 100% 3427/3427 [00:10<00:00, 313.98 examples/s]
Filter: 100% 447/447 [00:01<00:00, 284.90 examples/s]
train=2784  val=440  vocab=47
config.json: 100% 1.57k/1.57k [00:00<00:00, 5.13MB/s]

pytorch_model.bin: downloading bytes:  12% 154M/1.27G [00:01<00:07, 150MB/s, 13.4MB/s  ]  
pytorch_model.bin: downloading bytes:  14% 177M/1.27G [00:01<00:07, 139MB/s, 15.5MB/s  ]
pytorch_model.bin: reconstructing file:  18% 229M/1.27G [00:02<00:08, 122MB/s, 17.1MB/s  ]
pytorch_model.bin: downloading bytes:  20% 252M/1.27G [00:02<00:05, 180MB/s, 20.1MB/s  ] ]
pytorch_model.bin: downloading bytes:  35% 445M/1.27G [00:03<00:03, 236MB/s, 36.7MB/s  ] ]
pytorch_model.bin: downloading bytes:  40% 503M/1.27G [00:03<00:04, 175MB/s, 41.4MB/s  ] ]
pytorch_model.bin: downloading bytes:  42% 539M/1.27G [00:03<00:05, 125MB/s, 43.7MB/s  ] ]
pytorch_model.bin: downloading bytes:  44% 559M/1.27G [00:04<00:06, 114MB/s, 44.5MB/s  ] ]
pytorch_model.bin: downloading bytes:  49% 626M/1.27G [00:04<00:03, 172MB/s, 48

: 

## 8. Resume after a disconnect

Re-run cells 1-6, then run this instead of cell 7. It picks up from the last
checkpoint on Drive.

In [12]:
import shutil, os
shutil.move(os.path.join(CKPT_DIR, "checkpoint-800"),
            os.path.join(CKPT_DIR, "broken-checkpoint-800"))
print(sorted(os.listdir(CKPT_DIR)))

['broken-checkpoint-800', 'checkpoint-400']


In [ ]:
!cd src && python -u train.py \
    --output-dir "{CKPT_DIR}" \
    --max-steps 2100 \
    --batch-size 2 \
    --grad-accum 8 \
    --hub-model-id pnawani/welsh-asr-xlsr-300m \
    --push-to-hub \
    --resume

train=2784  val=440  vocab=47
Loading weights: 100% 422/422 [00:00<00:00, 27819.63it/s]
[transformers] Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
lm_head.weight               | MISSING    | 
lm_head.bias                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config an

## 9. Evaluate the fine-tuned model on the test set

Writes `results/finetuned_results.json`. Download it and run
`src/error_analysis.py` locally against it plus the Whisper baseline.

In [ ]:
!cd src && python -u evaluate.py \
    --model "{CKPT_DIR}" \
    --out ../results/finetuned_results.json \
    --progress-every 50

In [ ]:
from google.colab import files
files.download("/content/welsh-asr/results/finetuned_results.json")